In [6]:
# ─────────────────────────────────────────────
# Part 1. 라이브러리 설치 및 임포트
# ─────────────────────────────────────────────
# 터미널에서 먼저 실행
# pip install sentence-transformers langchain-huggingface langchain-community faiss-cpu

import json
import numpy as np
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document

print("라이브러리 임포트 완료")

C:\Users\rshyu\miniconda3\envs\aistudy_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


라이브러리 임포트 완료


In [7]:
# ─────────────────────────────────────────────
# Part 2. 청크 파일 로드
# ─────────────────────────────────────────────
# 이전 단계에서 만든 ingredient_chunks.json 불러오기

with open(r"C:\lecture\flow\00_data\02_processed\ingredient_chunks.json", "r", encoding="utf-8") as f:
    all_chunks = json.load(f)

print(f"총 청크 수: {len(all_chunks)}")
print(f"\n[샘플 확인]")
print(f"page_content: {all_chunks[0]['page_content']}")
print(f"metadata: {all_chunks[0]['metadata']}")

총 청크 수: 306836

[샘플 확인]
page_content: [(20S)-프로토파낙사다이올] EWG 스코어: 0 / 데이터 등급: 0
metadata: {'ingredient_ko': '(20S)-프로토파낙사다이올', 'ingredient_en': '(20S)-Protopanaxadiol', 'coos_score': '0', 'hw_ewg': None, 'is_allergy': None, 'pc_rating': None, 'chunk_type': 'ewg', 'chunk_weight': 1.0, 'source': 'coos'}


In [8]:
# ─────────────────────────────────────────────
# Part 3. LangChain Document 객체로 변환
# ─────────────────────────────────────────────
# FAISS는 LangChain의 Document 형식을 받아야 함
# Document = page_content(텍스트) + metadata(딕셔너리) 로 구성된 객체

docs = [
    Document(
        page_content=chunk["page_content"],
        metadata=chunk["metadata"]
    )
    for chunk in all_chunks
]

print(f"Document 변환 완료: {len(docs)}개")
print(f"\n[Document 샘플]")
print(f"page_content: {docs[0].page_content}")
print(f"metadata: {docs[0].metadata}")

Document 변환 완료: 306836개

[Document 샘플]
page_content: [(20S)-프로토파낙사다이올] EWG 스코어: 0 / 데이터 등급: 0
metadata: {'ingredient_ko': '(20S)-프로토파낙사다이올', 'ingredient_en': '(20S)-Protopanaxadiol', 'coos_score': '0', 'hw_ewg': None, 'is_allergy': None, 'pc_rating': None, 'chunk_type': 'ewg', 'chunk_weight': 1.0, 'source': 'coos'}


In [9]:
# ─────────────────────────────────────────────
# Part 4. 임베딩 모델 로드
# ─────────────────────────────────────────────
# jhgan/ko-sroberta-multitask
# - 한국어 특화 문장 임베딩 모델
# - 처음 실행 시 HuggingFace에서 자동 다운로드 (약 400MB)
# - 두 번째 실행부터는 로컬 캐시에서 바로 로드

embedding_model = HuggingFaceEmbeddings(
    model_name="jhgan/ko-sroberta-multitask",
    model_kwargs={"device": "cpu"},   # GPU 없으면 cpu, 있으면 "cuda"
    encode_kwargs={"normalize_embeddings": True},  # 코사인 유사도 계산을 위해 정규화
)

print("임베딩 모델 로드 완료")

# 테스트: 샘플 텍스트 벡터로 변환해보기
test_vector = embedding_model.embed_query("나이아신아마이드 EWG 등급")
print(f"벡터 차원 수: {len(test_vector)}")  # 768차원이면 정상

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 39804.78it/s]
RobertaModel LOAD REPORT from: jhgan/ko-sroberta-multitask
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


임베딩 모델 로드 완료
벡터 차원 수: 768


In [10]:
pip install tqdm

Note: you may need to restart the kernel to use updated packages.


In [11]:
# ─────────────────────────────────────────────
# Part 5. FAISS 인덱스 구축
# ─────────────────────────────────────────────
from tqdm import tqdm

BATCH_SIZE = 500  # 한 번에 처리할 청크 수

print(f"총 청크 수: {len(docs)}개")
print(f"배치 크기: {BATCH_SIZE}개씩 처리")
print("FAISS 인덱스 구축 시작...\n")

# 첫 번째 배치로 vectorstore 초기화
vectorstore = FAISS.from_documents(
    documents=docs[:BATCH_SIZE],
    embedding=embedding_model,
)

# 나머지 배치 순차 처리
batches = [docs[i:i+BATCH_SIZE] for i in range(BATCH_SIZE, len(docs), BATCH_SIZE)]

for batch in tqdm(batches, desc="FAISS 인덱스 구축", unit="배치"):
    batch_vectorstore = FAISS.from_documents(
        documents=batch,
        embedding=embedding_model,
    )
    vectorstore.merge_from(batch_vectorstore)  # 기존 인덱스에 병합

print(f"\nFAISS 인덱스 구축 완료!")
print(f"저장된 벡터 수: {vectorstore.index.ntotal}")

총 청크 수: 306836개
배치 크기: 500개씩 처리
FAISS 인덱스 구축 시작...



FAISS 인덱스 구축: 100%|██████████| 613/613 [8:31:47<00:00, 50.09s/배치]  


FAISS 인덱스 구축 완료!
저장된 벡터 수: 306836


In [13]:
# ─────────────────────────────────────────────
# Part 6. FAISS 인덱스 로컬 저장
# ─────────────────────────────────────────────
# 매번 임베딩 다시 하면 시간이 오래 걸리니까
# 한 번 만든 인덱스를 로컬에 저장해두고 재사용

FAISS_PATH = r"C:\lecture\flow\00_data\02_processed\faiss_index"

vectorstore.save_local(FAISS_PATH)
print(f"FAISS 인덱스 저장 완료: {FAISS_PATH}")

# 저장되는 파일 2개
# - faiss_index/index.faiss  → 벡터 데이터
# - faiss_index/index.pkl    → 메타데이터

FAISS 인덱스 저장 완료: C:\lecture\flow\00_data\02_processed\faiss_index
